# 객체 탐지 CPU 기준 모델: 패치 학습·슬라이딩 윈도·NMS

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

작고 일정한 크기·외형의 객체를 위한 학습 가능한 단순 탐지기입니다. 일반 객체 탐지 성능을 보장하지 않습니다. 큰 이미지는 윈도 수가 급증하므로 축소·stride를 조정하세요.

In [ ]:
DEMO=True
TRAIN_MANIFEST='data/detection_train.csv';TEST_MANIFEST='data/detection_test.csv'
ANNOTATIONS='data/boxes.csv' # id,class_id,xmin,ymin,xmax,ymax (픽셀 xyxy)
IMAGE_ROOT='data';ID='id';PATH_COL='path';GROUP=None
WINDOW_SIZES=[12,16,20];STRIDE=4;PATCH_SIZE=(8,8)
NEGATIVES_PER_IMAGE=8;CONFIDENCE=.75;NMS_IOU=.3;MATCH_IOU=.5
OUTPUT='outputs/image/detections.csv'


## 공통 함수

좌표는 픽셀 xyxy이며 xmax,ymax는 crop 끝 좌표입니다.

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report,
    ConfusionMatrixDisplay, silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.dummy import DummyClassifier, DummyRegressor
SEED=42
rng=np.random.default_rng(SEED)

def read_table(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower()=='.parquet': return pd.read_parquet(p)
    return pd.read_csv(p, sep='\t' if p.suffix.lower()=='.tsv' else ',')

def check_ids(df, id_col):
    if id_col not in df or df[id_col].isna().any() or df[id_col].duplicated().any():
        raise ValueError(f'{id_col}: 각 예측 단위에 결측 없는 고유 ID가 필요합니다.')

def split_rows(df, target=None, task='classification', strategy='random', group=None, time_col=None, fraction=.25, gap=0):
    idx=np.arange(len(df))
    if not 0<fraction<1: raise ValueError('validation fraction은 0~1 사이여야 합니다.')
    if strategy=='group':
        if not group or df[group].isna().any(): raise ValueError('유효한 그룹 열이 필요합니다.')
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=SEED).split(df,groups=df[group]))
        assert set(df.iloc[a][group]).isdisjoint(set(df.iloc[b][group]))
    elif strategy=='time':
        if not time_col: raise ValueError('시간 열을 지정하세요.')
        t=pd.to_datetime(df[time_col],errors='raise')
        if t.isna().any(): raise ValueError('시간 결측을 해결하세요.')
        unique=np.sort(t.unique());cut=int(len(unique)*(1-fraction))
        if cut<=gap or cut>=len(unique): raise ValueError('시간 분할에 필요한 데이터가 부족합니다.')
        a=idx[t<unique[cut-gap]];b=idx[t>=unique[cut]]
        assert t.iloc[a].max()<t.iloc[b].min()
    elif strategy in ('random','stratified'):
        strat=df[target] if task=='classification' and target else None
        if strat is not None and strat.value_counts().min()<2:
            raise ValueError('표본 1개인 클래스가 있습니다. 병합/수집/분할 정책을 검토하세요.')
        a,b=train_test_split(idx,test_size=fraction,random_state=SEED,stratify=strat)
    else: raise ValueError(f'지원하지 않는 분할: {strategy}')
    if not len(a) or not len(b): raise ValueError('빈 학습/검증 분할')
    if task=='classification' and target and set(df.iloc[b][target])-set(df.iloc[a][target]):
        raise ValueError('학습에 없는 클래스가 검증에 존재합니다.')
    return np.asarray(a),np.asarray(b)

def clean_tabular(X):
    out=X.copy()
    for c in out:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c]=pd.to_numeric(out[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        else: out[c]=out[c].map(lambda v: str(v) if pd.notna(v) else np.nan)
    return out

def tabular_preprocessor(X, robust=False):
    num=X.select_dtypes(include='number').columns.tolist()
    cat=[c for c in X if c not in num]
    parts=[]
    if num: parts.append(('num',make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RobustScaler() if robust else StandardScaler()),num))
    if cat: parts.append(('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='__MISSING__',keep_empty_features=True),OneHotEncoder(handle_unknown='ignore',min_frequency=2)),cat))
    if not parts: raise ValueError('특징 열이 없습니다.')
    return ColumnTransformer(parts)

def metrics_for(model,X,y,task):
    pred=model.predict(X)
    if task=='regression': return {'mae':float(mean_absolute_error(y,pred)),'rmse':float(np.sqrt(mean_squared_error(y,pred))),'r2':float(r2_score(y,pred))}
    out={'accuracy':float(accuracy_score(y,pred)),'f1_macro':float(f1_score(y,pred,average='macro',zero_division=0))}
    if hasattr(model,'predict_proba'):
        p=model.predict_proba(X);classes=model.classes_
        out['log_loss']=float(log_loss(y,p,labels=classes))
        if len(classes)==2 and len(np.unique(y))==2: out['roc_auc']=float(roc_auc_score(np.asarray(y)==classes[1],p[:,1]))
    return out

def fit_compare(candidates,X,y,a,b,task,metric,budget=120):
    direction={'accuracy':True,'f1_macro':True,'roc_auc':True,'r2':True,'log_loss':False,'mae':False,'rmse':False}
    if metric not in direction: raise ValueError('지원 지표를 선택하거나 metrics_for를 확장하세요.')
    t0=time.monotonic();rows=[];fitted={}
    for name,estimator in candidates.items():
        if rows and time.monotonic()-t0>=budget: break
        start=time.monotonic();model=clone(estimator).fit(X.iloc[a] if hasattr(X,'iloc') else X[a],y.iloc[a] if hasattr(y,'iloc') else y[a])
        stats=metrics_for(model,X.iloc[b] if hasattr(X,'iloc') else X[b],y.iloc[b] if hasattr(y,'iloc') else y[b],task)
        if metric not in stats or not np.isfinite(stats[metric]): raise ValueError(f'{metric}: 이 분할/모델에서 계산할 수 없습니다.')
        rows.append({'model':name,**stats,'seconds':time.monotonic()-start});fitted[name]=model
    table=pd.DataFrame(rows).sort_values(metric,ascending=not direction[metric]);display(table)
    best=table.iloc[0]['model']
    return best,fitted[best],table

def write_submission(ids,pred,id_col,target_cols,output,sample_path=None,probabilities=False):
    ids=pd.Series(ids).reset_index(drop=True)
    arr=np.asarray(pred)
    if arr.ndim==1: arr=arr[:,None]
    if arr.ndim!=2 or arr.shape!=(len(ids),len(target_cols)): raise ValueError('예측의 행/열 수와 제출 규격이 다릅니다.')
    if ids.isna().any() or ids.duplicated().any(): raise ValueError('제출 ID 결측/중복')
    if len(set(target_cols))!=len(target_cols) or id_col in target_cols: raise ValueError('제출 열 이름 중복')
    out=pd.DataFrame(arr,columns=target_cols);out.insert(0,id_col,ids.to_numpy())
    if out.isna().any().any(): raise ValueError('제출 값 결측')
    numeric=out[target_cols].select_dtypes(include='number')
    if numeric.size and not np.isfinite(numeric.to_numpy()).all(): raise ValueError('제출 값 무한대')
    if probabilities:
        v=arr.astype(float)
        if not np.isfinite(v).all() or (v<0).any() or (v>1).any(): raise ValueError('확률 범위 오류')
        if v.shape[1]>1 and not np.allclose(v.sum(axis=1),1,atol=1e-5): raise ValueError('확률 합 오류')
    if sample_path:
        sample=read_table(sample_path);check_ids(sample,id_col)
        if set(sample.columns)!=set(out.columns): raise ValueError('sample_submission과 열이 다릅니다.')
        if len(sample)!=len(out) or set(sample[id_col])!=set(out[id_col]): raise ValueError('sample_submission과 ID가 다릅니다. ID 자료형도 확인하세요.')
        out=sample[[id_col]].merge(out,on=id_col,how='left',validate='one_to_one')[sample.columns]
    p=Path(output);p.parent.mkdir(parents=True,exist_ok=True);out.to_csv(p,index=False)
    display(out.head());print('저장:',p,'형태:',out.shape)
    return out

def classification_output(model,X,kind='label',order=None,positive=None):
    if kind=='label': return model.predict(X)
    if not hasattr(model,'predict_proba'): raise ValueError('확률을 지원하는 모델이 필요합니다.')
    classes=list(model.classes_);p=model.predict_proba(X)
    if kind=='positive_probability':
        if positive not in classes: raise ValueError('positive_class를 실제 클래스 값으로 지정하세요.')
        return p[:,classes.index(positive)]
    if kind!='probability' or order is None or len(order)!=len(classes) or set(order)!=set(classes):
        raise ValueError('공식 제출 열에 대응하는 class_order를 지정하세요.')
    return p[:,[classes.index(v) for v in order]]


## 이미지·박스 입력

train manifest와 annotation CSV를 ID로 연결합니다. 같은 영상의 프레임은 GROUP으로 분리하세요.

In [ ]:
from PIL import Image, ImageDraw
if DEMO:
    root=Path('outputs/detection_demo');root.mkdir(parents=True,exist_ok=True)
    rows=[];annotations=[]
    for i in range(48):
        arr=rng.integers(0,20,(40,40,3),dtype=np.uint8)
        x,y=rng.integers(1,24,size=2);w=16
        arr[y:y+w,x:x+w,0]=230;arr[y:y+w,x:x+w,1:]=30
        p=root/f'{i}.png';Image.fromarray(arr).save(p)
        rows.append({ID:i,PATH_COL:str(p.resolve())})
        annotations.append({ID:i,'class_id':0,'xmin':x,'ymin':y,'xmax':x+w,'ymax':y+w})
    train=pd.DataFrame(rows[:36]);test=pd.DataFrame(rows[36:]);boxes=pd.DataFrame(annotations[:36])
else:
    train=read_table(TRAIN_MANIFEST);test=read_table(TEST_MANIFEST);boxes=read_table(ANNOTATIONS)
    for frame in (train,test): frame[PATH_COL]=frame[PATH_COL].map(lambda p:str(Path(IMAGE_ROOT)/str(p)))
check_ids(train,ID);check_ids(test,ID)
if not set(boxes[ID])<=set(train[ID]): raise ValueError('annotation ID가 train manifest에 없습니다.')
if boxes.empty or boxes.isna().any().any(): raise ValueError('annotation이 비어 있거나 결측입니다.')
if not np.all(boxes.class_id.to_numpy()==boxes.class_id.astype(int).to_numpy()) or (boxes.class_id<0).any(): raise ValueError('class_id는 0 이상의 정수여야 합니다.')
classes=sorted(boxes.class_id.unique().tolist())
label_map={value:i+1 for i,value in enumerate(classes)} # 0은 배경

def load_image(path):
    with Image.open(path) as im:
        # bbox 좌표가 저장 파일 기준이므로 EXIF 회전을 자동 적용하지 않음
        return np.asarray(im.convert('RGB')).copy()
images={r[ID]:load_image(r[PATH_COL]) for _,r in train.iterrows()}
for _,box in boxes.iterrows():
    height,width=images[box[ID]].shape[:2]
    if not 0<=box.xmin<box.xmax<=width or not 0<=box.ymin<box.ymax<=height: raise ValueError('이미지 밖/잘못된 bbox')


## 패치 학습·홀드아웃 평가·재학습·탐지

평가는 고정 임계값에서의 precision/recall이며 mAP가 아닙니다. 결과는 범용 long CSV로, 공식 제출 형식에 맞춰 변환해야 합니다. 미탐지 이미지는 detection_counts.csv에 0으로 남습니다.

In [ ]:
def iou(a,b):
    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])
    intersection=max(0,x2-x1)*max(0,y2-y1)
    return intersection/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-intersection+1e-12)

def patch_feature(arr):
    return np.asarray(Image.fromarray(arr).resize(PATCH_SIZE),dtype=float).ravel()/255

def patches(ids):
    X=[];y=[]
    for image_id in ids:
        arr=images[image_id];h,w=arr.shape[:2];ground=boxes[boxes[ID]==image_id]
        for _,b in ground.iterrows():
            x1,y1,x2,y2=map(int,[b.xmin,b.ymin,b.xmax,b.ymax])
            X.append(patch_feature(arr[y1:y2,x1:x2]));y.append(label_map[b.class_id])
        count=0
        for _ in range(NEGATIVES_PER_IMAGE*50):
            side=int(rng.choice(WINDOW_SIZES))
            if side>min(h,w): continue
            x=int(rng.integers(0,w-side+1));yy=int(rng.integers(0,h-side+1));candidate=[x,yy,x+side,yy+side]
            if any(iou(candidate,[b.xmin,b.ymin,b.xmax,b.ymax])>.05 for _,b in ground.iterrows()): continue
            X.append(patch_feature(arr[yy:yy+side,x:x+side]));y.append(0);count+=1
            if count>=NEGATIVES_PER_IMAGE: break
    if 0 not in y or len(set(y))<2: raise ValueError('배경/객체 패치를 충분히 확보하세요.')
    return np.asarray(X),np.asarray(y)

def detect(model,arr):
    h,w=arr.shape[:2];candidates=[];vectors=[]
    for side in WINDOW_SIZES:
        if side>min(h,w): continue
        xs=sorted(set(list(range(0,w-side+1,STRIDE))+[w-side]))
        ys=sorted(set(list(range(0,h-side+1,STRIDE))+[h-side]))
        for yy in ys:
            for x in xs:
                candidates.append([x,yy,x+side,yy+side]);vectors.append(patch_feature(arr[yy:yy+side,x:x+side]))
    if not vectors: return []
    probabilities=model.predict_proba(np.asarray(vectors));detections=[]
    for window,p in zip(candidates,probabilities):
        j=int(np.argmax(p));label=model.classes_[j]
        if label!=0 and p[j]>=CONFIDENCE: detections.append({'class_id':classes[int(label)-1],'confidence':float(p[j]),'box':window})
    kept=[]
    for item in sorted(detections,key=lambda d:d['confidence'],reverse=True):
        if not any(item['class_id']==k['class_id'] and iou(item['box'],k['box'])>NMS_IOU for k in kept): kept.append(item)
    return kept

if STRIDE<1 or any(v<1 for v in WINDOW_SIZES): raise ValueError('stride/window는 양수')
a,b=split_rows(train,task='regression',strategy='group' if GROUP else 'random',group=GROUP)
X,y=patches(train.iloc[a][ID]);model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,class_weight='balanced',random_state=SEED)).fit(X,y)
tp=fp=fn=0
for image_id in train.iloc[b][ID]:
    truth=boxes[boxes[ID]==image_id];matched=set()
    for d in detect(model,images[image_id]):
        eligible=[(idx,iou(d['box'],[r.xmin,r.ymin,r.xmax,r.ymax])) for idx,r in truth.iterrows() if idx not in matched and r.class_id==d['class_id']]
        if eligible and max(v for _,v in eligible)>=MATCH_IOU:
            winner=max(eligible,key=lambda p:p[1])[0];matched.add(winner);tp+=1
        else:fp+=1
    fn+=len(truth)-len(matched)
print({'precision_at_fixed_threshold':tp/max(1,tp+fp),'recall_at_fixed_threshold':tp/max(1,tp+fn),'TP':tp,'FP':fp,'FN':fn})
X,y=patches(train[ID]);final_model=clone(model).fit(X,y)
records=[];counts=[]
for _,row in test.iterrows():
    detections=detect(final_model,load_image(row[PATH_COL]));counts.append({ID:row[ID],'detection_count':len(detections)})
    for d in detections: records.append({ID:row[ID],'class_id':d['class_id'],'confidence':d['confidence'],**dict(zip(['xmin','ymin','xmax','ymax'],d['box']))})
submission=pd.DataFrame(records,columns=[ID,'class_id','confidence','xmin','ymin','xmax','ymax'])
p=Path(OUTPUT);p.parent.mkdir(parents=True,exist_ok=True);submission.to_csv(p,index=False)
pd.DataFrame(counts).to_csv(p.with_name('detection_counts.csv'),index=False)
display(submission.head());display(pd.DataFrame(counts).head())
